# 03 — Held-out judging

Plan and, only when explicitly enabled, judge held-out responses with stored rubrics. Complete the rubric canary in notebook 02 first. The default controls perform zero model calls.

In [ ]:
from pathlib import Path

import pandas as pd

from rubric_order.api import (
    find_project_root,
    judge_responses,
    load_experiment_config,
    plan_judging,
    summarize_judgments,
)

PROJECT_ROOT = find_project_root(Path.cwd())

In [ ]:
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot.yaml"
RUN_ID = "pilot-001"
RUNTIME_OVERRIDES = {}

EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
SELECTED_METHODS = ["sequential", "batch", "static"]
SELECTED_BLOCKS = [0, 1]
SELECTED_ORDERS = ["ascending", "descending", "random", "reverse_random"]
SELECTED_REPLICATES = [0]

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
selection = {
    "methods": SELECTED_METHODS,
    "blocks": SELECTED_BLOCKS,
    "orders": SELECTED_ORDERS,
    "replicates": SELECTED_REPLICATES,
}
plan = plan_judging(config, RUN_ID, **selection)
{
    "model": plan["model"],
    "ready": plan["ready"],
    "valid_rubrics": plan["valid_rubrics"],
    "missing_rubrics": plan["missing_rubrics"],
    "invalid_rubrics": plan["invalid_rubrics"],
    "planned_calls": plan["planned_calls"],
    "estimated_max_generated_tokens": plan["estimated_max_generated_tokens"],
    "execution_enabled": EXECUTE_MODEL_CALLS,
    "max_calls": MAX_CALLS,
}

In [ ]:
before = summarize_judgments(config, RUN_ID, **selection)
before

In [ ]:
def report_progress(update):
    print(
        f"{update['rubrics_completed']}/{update['rubrics_total']} rubrics: "
        f"{update['method']} block {update['block_id']} {update['order']} "
        f"replicate {update['replicate']}"
    )


if EXECUTE_MODEL_CALLS:
    judging_result = judge_responses(
        config,
        RUN_ID,
        execute_model_calls=True,
        max_calls=MAX_CALLS,
        progress_callback=report_progress,
        **selection,
    )
else:
    judging_result = {**plan, "executed": False, "actual_calls": 0}
    print("No files changed. Inspect the in-memory plan before enabling execution.")
{
    key: judging_result.get(key)
    for key in [
        "executed",
        "ready",
        "planned_calls",
        "actual_calls",
        "cache_hits",
        "dry_run_manifest",
    ]
}

In [ ]:
after = summarize_judgments(config, RUN_ID, **selection)
{
    key: after[key]
    for key in [
        "completed",
        "cached",
        "failed",
        "remaining",
        "schema_failures",
        "retries",
        "prompt_eval_tokens",
        "eval_tokens",
        "latency_seconds",
        "score_distribution",
    ]
}

In [ ]:
pd.DataFrame(after["sample"])

Rerunning the execution cell with the same selection reuses successful cached responses. Failed or missing responses remain visible in the summaries; complete held-out response text is not displayed.